# 01 — Bottom-up feedback cycle only

Questo notebook esegue **solo l'estensione bottom-up**. Non riesegue actor extraction, HLG extraction o LLG extraction iniziale.

Flusso:

1. legge i JSON già prodotti da `output/top_down/`;
2. ricostruisce gli oggetti `HighLevelGoals` e `LowLevelGoals` salvati dalla baseline;
3. avvia `run_global_goal_cycle` usando i callback del generatore/evaluator top-down originale per le sole correzioni richieste dal ciclo;
4. salva un nuovo JSON in `output/top_down_bottom_up/`;
5. lascia immutato il file top-down originale.

In questo modo il confronto sperimentale è davvero A/B: **top-down** contro **top-down + bottom-up**, a partire dallo stesso identico stato iniziale.

In [1]:
from pathlib import Path
import gc
import json
import re
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_ROOT = PROJECT_ROOT / "output"
TOP_DOWN_DIR = OUTPUT_ROOT / "top_down"
TOP_DOWN_BOTTOM_UP_DIR = OUTPUT_ROOT / "top_down_bottom_up"
BOTTOM_UP_INPUT_DIR = OUTPUT_ROOT / "bottom_up_inputs"
ITERATIONS_ROOT = OUTPUT_ROOT / "bottom_up_iterations"

for path in (
    TOP_DOWN_DIR,
    TOP_DOWN_BOTTOM_UP_DIR,
    BOTTOM_UP_INPUT_DIR,
    ITERATIONS_ROOT,
):
    path.mkdir(parents=True, exist_ok=True)

# Optional: insert explicit Path(...) values here to run only selected baselines.
BASELINE_FILE_OVERRIDES = []

# Maximum number of feedback iterations. By default use the orchestrator constant.
GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE = None

# One remote request cannot keep the notebook busy forever.
REQUEST_TIMEOUT_SECONDS = 120.0
API_MAX_RETRIES = 1

print("Project root:", PROJECT_ROOT)
print("Top-down inputs:", TOP_DOWN_DIR)
print("Top-down + bottom-up outputs:", TOP_DOWN_BOTTOM_UP_DIR)

Project root: C:\Users\agnes_lryeu3v\Desktop\tesi
Top-down inputs: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down
Top-down + bottom-up outputs: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down_bottom_up


## 1. Individuazione degli output top-down

Il notebook usa tutti i file `*_top_down.json` presenti nella cartella della baseline, salvo override espliciti.

In [2]:
if BASELINE_FILE_OVERRIDES:
    baseline_files = [Path(p) for p in BASELINE_FILE_OVERRIDES]
else:
    baseline_files = sorted(TOP_DOWN_DIR.glob("*_top_down.json"))

missing = [p for p in baseline_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing baseline files: {missing}")
if not baseline_files:
    raise FileNotFoundError(
        f"No *_top_down.json found in {TOP_DOWN_DIR}. "
        "Run 01_pipeline_execution_top_down_only.ipynb first."
    )

print(f"Found {len(baseline_files)} top-down baseline file(s):")
for p in baseline_files:
    print("-", p.name)

Found 4 top-down baseline file(s):
- Genome Nexus_ZERO_SHOT_feedback_original_top_down_top_down.json
- Gestao Hospital_ZERO_SHOT_feedback_original_top_down_top_down.json
- London Ambulance Service_ZERO_SHOT_feedback_original_top_down_top_down.json
- SIA Project 25 26_ZERO_SHOT_feedback_original_top_down_top_down.json


## 2. Caricamento dello stato iniziale

I nuovi file top-down contengono direttamente `structuredHighLevelGoals` e `structuredLowLevelGoals`, quindi il ciclo riusa le relazioni padre-figlio effettivamente prodotte dalla top-down.

Per compatibilità con vecchi output privi di tali campi, è presente un fallback a `low_level_goal_mapper.py`. Il fallback può però richiedere una `BRANCH_SPECIFICATION` hard-coded compatibile con quello specifico output; per nuovi esperimenti è preferibile rigenerare la baseline con il notebook top-down separato.

In [3]:
from src.data_model import HighLevelGoals, LowLevelGoals
from src.bottom_up.low_level_goal_mapper import map_low_level_goals, load_mapped_bottom_up_input
from src.examples.shot_learning import ShotPromptingMode


def _infer_mode_from_name(path: Path) -> str:
    match = re.search(r"(?:^|_)(ZERO_SHOT|ONE_SHOT|FEW_SHOT)(?=_|$)", path.stem.upper())
    if not match:
        raise ValueError(f"Cannot infer prompting mode from {path.name}")
    return match.group(1)


def load_top_down_state(baseline_file: Path):
    with baseline_file.open("r", encoding="utf-8") as f:
        payload = json.load(f)

    description = payload["description"]
    config = payload.get("pipelineConfiguration") or {}
    prompting_mode = config.get("promptingMode") or _infer_mode_from_name(baseline_file)
    llama_ablation = bool(config.get("llamaAblation", "NOLLAMA" in baseline_file.stem.upper().replace("_", "")))

    structured_hlg = payload.get("structuredHighLevelGoals")
    structured_llg = payload.get("structuredLowLevelGoals")

    if structured_hlg is not None and structured_llg is not None:
        high_level_goals = HighLevelGoals.model_validate(structured_hlg)
        low_level_goals = LowLevelGoals.model_validate(structured_llg)
        source_kind = "structured baseline fields"
        mapped_input_file = None
    else:
        mapped_input_file = BOTTOM_UP_INPUT_DIR / f"{baseline_file.stem}_bottom_up_input.json"
        map_low_level_goals(
            source_file=baseline_file,
            destination_file=mapped_input_file,
        )
        (
            description,
            high_level_goals,
            low_level_goals,
        ) = load_mapped_bottom_up_input(mapped_input_file)
        source_kind = "legacy deterministic mapper"

    return {
        "payload": payload,
        "description": description,
        "high_level_goals": high_level_goals,
        "low_level_goals": low_level_goals,
        "prompting_mode": ShotPromptingMode[prompting_mode],
        "llama_ablation": llama_ablation,
        "source_kind": source_kind,
        "mapped_input_file": mapped_input_file,
    }

## 3. Callback di correzione e lifecycle dei client

Quando il Global Goal Evaluator richiede una correzione, il ciclo richiama i callback già implementati in `extractor.py`. Questi callback passano attraverso il generatore top-down originale e il relativo evaluator/reflection loop; non viene introdotto un secondo generatore alternativo.

In [4]:
from openai import OpenAI
from key import get_key_openai, get_key_llama
from src.extraction.extractor import build_bottom_up_evaluated_generation_callbacks
from src.bottom_up.goal_cycle_orchestrator import (
    run_global_goal_cycle,
    DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS,
)


def _close_client_safely(client):
    if client is None:
        return
    try:
        client.close()
    except Exception as exc:
        print(f"Warning while closing client: {type(exc).__name__}: {exc}")


def reset_llm_clients():
    import src.llm_clients as llm_clients

    _close_client_safely(getattr(llm_clients, "client", None))
    _close_client_safely(getattr(llm_clients, "llama", None))

    llm_clients.client = OpenAI(
        api_key=get_key_openai(),
        timeout=REQUEST_TIMEOUT_SECONDS,
        max_retries=API_MAX_RETRIES,
    )
    llm_clients.llama = OpenAI(
        api_key=get_key_llama(),
        base_url="https://api.groq.com/openai/v1",
        timeout=REQUEST_TIMEOUT_SECONDS,
        max_retries=API_MAX_RETRIES,
    )


def close_llm_clients():
    module = sys.modules.get("src.llm_clients")
    if module is not None:
        _close_client_safely(getattr(module, "client", None))
        _close_client_safely(getattr(module, "llama", None))
    gc.collect()


def combined_output_file_for(baseline_file: Path) -> Path:
    stem = baseline_file.stem
    base = stem.removesuffix("_top_down")
    return TOP_DOWN_BOTTOM_UP_DIR / f"{base}_top_down_bottom_up.json"


def iteration_directory_for(baseline_file: Path) -> Path:
    return ITERATIONS_ROOT / baseline_file.stem.removesuffix("_top_down")

## 4. Esecuzione bottom-up

Ogni dataset viene processato in modo indipendente. L'output combinato parte da una copia del JSON top-down e aggiunge il trace completo del ciclo e le collezioni finali. Il file top-down di partenza non viene mai sovrascritto.

In [5]:
cycle_results = {}
cycle_errors = {}

max_iterations = (
    DEFAULT_GLOBAL_CYCLE_MAX_ITERATIONS
    if GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE is None
    else int(GLOBAL_CYCLE_MAX_ITERATIONS_OVERRIDE)
)

for baseline_file in baseline_files:
    try:
        state = load_top_down_state(baseline_file)
    except Exception as exc:
        dataset_name = baseline_file.stem
        cycle_errors[dataset_name] = f"INPUT_LOAD_ERROR - {type(exc).__name__}: {exc}"
        print(f"\n[{dataset_name}] FAILED while loading top-down state: {exc}")
        continue

    baseline_payload = state["payload"]
    dataset_name = baseline_payload["name"]
    iteration_dir = iteration_directory_for(baseline_file)
    iteration_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{dataset_name}] starting bottom-up cycle")
    print("  input:", baseline_file.name)
    print("  state source:", state["source_kind"])
    print("  prompting mode:", state["prompting_mode"].name)
    print("  evaluator ablation:", state["llama_ablation"])
    print("  initial HLG:", len(state["high_level_goals"].goals))
    print("  initial LLG:", len(state["low_level_goals"].low_level_goals))

    generate_hlg, regenerate_llg = build_bottom_up_evaluated_generation_callbacks(
        project_description=state["description"],
        mode=state["prompting_mode"],
        evaluator_ablation=state["llama_ablation"],
    )

    reset_llm_clients()
    started_at = time.perf_counter()
    try:
        result = run_global_goal_cycle(
            project_description=state["description"],
            initial_high_level_goals=state["high_level_goals"],
            initial_low_level_goals=state["low_level_goals"],
            generate_high_level_goals=generate_hlg,
            regenerate_low_level_goals=regenerate_llg,
            evaluation_output_directory=iteration_dir,
            max_iterations=max_iterations,
        )
    except Exception as exc:
        cycle_errors[dataset_name] = f"{type(exc).__name__}: {exc}"
        print(f"  FAILED: {cycle_errors[dataset_name]}")
        continue
    finally:
        close_llm_clients()
        print(f"  client resources closed after {time.perf_counter() - started_at:.1f}s")

    cycle_results[dataset_name] = result

    combined_payload = dict(baseline_payload)
    combined_payload.update(
        {
            "baselineSourceFile": baseline_file.name,
            "bottomUpInputSource": state["source_kind"],
            "bottomUpMappedInputFile": (
                str(state["mapped_input_file"]) if state["mapped_input_file"] else None
            ),
            "evaluationOutputDirectory": str(iteration_dir),
            "globalGoalCycle": result.model_dump(mode="json"),
            "finalHighLevelGoals": result.final_high_level_goals.model_dump(mode="json"),
            "finalLowLevelGoals": result.final_low_level_goals.model_dump(mode="json"),
        }
    )

    combined_file = combined_output_file_for(baseline_file)
    with combined_file.open("w", encoding="utf-8") as f:
        json.dump(combined_payload, f, indent=4, ensure_ascii=False)

    print(
        f"  saved: {combined_file.name} | "
        f"converged={result.converged}, "
        f"iterations={result.completed_iterations}, "
        f"HLG={len(result.final_high_level_goals.goals)}, "
        f"LLG={len(result.final_low_level_goals.low_level_goals)}"
    )


[Genome Nexus] starting bottom-up cycle
  input: Genome Nexus_ZERO_SHOT_feedback_original_top_down_top_down.json
  state source: structured baseline fields
  prompting mode: ZERO_SHOT
  evaluator ablation: False
  initial HLG: 3
  initial LLG: 8
Groq evaluator model not available; using fallback: qwen/qwen3.6-27b
Focused High Level Goal STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks.You're tasked to extract high level goals from a software description for each provided actor that is expected to interact with the software.Following the Goal-Oriented Requirements Engineering (GORE) frameworks, high-level goals are strategic objectives that define the 'why' behind a system. They are usually abstract, business-oriented, and independent of technical implementation. They represent the needs of stakeholders or the organization. Focus: Vision and justification. Generate ONLY the functional goals.

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\agnes_lryeu3v\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\agnes_lryeu3v\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\agnes_lryeu3v\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\agnes_lryeu3v\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\agnes_lryeu3v\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Focused High Level Goal STARTING... (attempt 1)
No feedback provided!
This is the provided sys prompt:  You are a helpful assistant expert in software engineering tasks.You're tasked to extract high level goals from a software description for each provided actor that is expected to interact with the software.Following the Goal-Oriented Requirements Engineering (GORE) frameworks, high-level goals are strategic objectives that define the 'why' behind a system. They are usually abstract, business-oriented, and independent of technical implementation. They represent the needs of stakeholders or the organization. Focus: Vision and justification. Generate ONLY the functional goals.

FOCUSED BOTTOM-UP REGENERATION:
Generate exactly ONE High-Level Goal representing the single stakeholder intention described in the supplied focused input. Keep it at WHY level and do not infer unrelated goals.

BOTTOM-UP CORRECTION GUIDANCE:
The reconstructed HLG fails to preserve the specific functional scope o

## 5. Confronto strutturale rapido

Questa sezione mostra solo le cardinalità prima/dopo e lo stato di convergenza. **Precision, Recall e F1 non vengono calcolati qui**: sono volutamente delegati al terzo notebook, che legge i due output persistiti in modo indipendente.

In [6]:
print("\nBottom-up execution summary")
print("---------------------------")
print(f"Inputs considered: {len(baseline_files)}")
print(f"Cycles completed: {len(cycle_results)}")
print(f"Cycles failed: {len(cycle_errors)}")
print(f"Output directory: {TOP_DOWN_BOTTOM_UP_DIR}")

for dataset_name, result in cycle_results.items():
    print(
        f"- {dataset_name}: converged={result.converged}, "
        f"stop_reason={result.stop_reason}, "
        f"iterations={result.completed_iterations}, "
        f"final HLG={len(result.final_high_level_goals.goals)}, "
        f"final LLG={len(result.final_low_level_goals.low_level_goals)}"
    )

if cycle_errors:
    print("\nErrors:")
    for dataset_name, error in cycle_errors.items():
        print(f"- {dataset_name}: {error}")


Bottom-up execution summary
---------------------------
Inputs considered: 4
Cycles completed: 4
Cycles failed: 0
Output directory: C:\Users\agnes_lryeu3v\Desktop\tesi\output\top_down_bottom_up
- Genome Nexus: converged=True, stop_reason=GlobalGoalCycleStopReason.ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED, iterations=3, final HLG=3, final LLG=11
- Gestao Hospital: converged=True, stop_reason=GlobalGoalCycleStopReason.ALL_BRANCHES_CONFIRMED_AND_DOCUMENTATION_COVERED, iterations=2, final HLG=5, final LLG=18
- London Ambulance Service: converged=False, stop_reason=GlobalGoalCycleStopReason.MAX_ITERATIONS_REACHED, iterations=5, final HLG=5, final LLG=17
- SIA Project 25 26: converged=False, stop_reason=GlobalGoalCycleStopReason.HIGH_LEVEL_REGENERATION_FAILED, iterations=1, final HLG=5, final LLG=18
